# 피부 특징 학습 — 모공·주름·색소·여드름·피부나이

부위 크롭 1장을 넣으면 그 부위의 피부 특징들을 한 번에 예측한다.
백본(ConvNeXt) 하나를 모든 부위·항목이 공유하고, 항목마다 작은 헤드가 붙는다.

**정답이 있는 칸만 학습한다.** 부위마다 있는 항목이 다르다.

| | 이마 | 미간 | 눈가 좌/우 | 볼 좌/우 |
|---|:-:|:-:|:-:|:-:|
| 모공 | · | · | · | 등급 + 개수 |
| 주름 | 등급 | 등급 | 등급 + Ra | · |
| 색소 | 등급 | · | · | 등급 |
| 여드름 | 개수 | · | · | 개수 |
| 피부나이 | 나이 | 나이 | 나이 | 나이 |

**한 샘플 = 크롭 1장.** 사람 단위로 학습하면 샘플이 800개뿐인데, 크롭 단위면 9,000개가 된다.
추론할 때 같은 (사람, 부위)의 여러 사진 예측을 평균하면 사람 단위 예측이 된다.

**나눌 때는 사람 단위.** 같은 사람의 사진은 정답이 같아서 섞이면 점수가 부풀려진다.

구성: ① 설정 → ② 데이터 → ③ 학습 함수 → **A. 하이퍼파라미터 실험** → **B. 최종 학습** → **C. 테스트 평가**

## ① 설정

실험은 이 `CFG`의 값을 바꿔가며 한다. 모델 구조만 `model.py`에 있다.

In [ ]:
import csv
import importlib
import json
import sys
import time
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset, Sampler, get_worker_info

import model as model_module

importlib.reload(model_module)
from model import SkinModel

# 전처리 모듈에서 캔버스 크기와 정규화 함수를 가져온다 (전처리와 학습이 같은 규칙을 쓰도록)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "data_preprocessing"))
import face_regions

importlib.reload(face_regions)
from face_regions import CANVAS, normalize_crop

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

CROP_DIR = ROOT / "data" / "01.원천데이터"
INDEX_PATH = ROOT / "data" / "index.csv"
LABEL_PATH = ROOT / "data" / "02.정답지데이터" / "labels_region.csv"
AILAB_DIR = ROOT / "ailabtools" / "test_set"
OUT_DIR = Path.cwd() / "outputs"
OUT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE == "cuda"  # NVIDIA에서는 혼합정밀도로 2배 빨라진다

REGIONS = ["forehead", "glabella", "l_eye", "r_eye", "l_cheek", "r_cheek"]

# 항목 → (정답 열, 정답이 있는 부위, log 변환 여부)
#   log=True: 개수처럼 한쪽으로 치우친 값은 log(1+x)로 바꿔 학습한다
TASKS = {
    "pore_count": dict(column="pore_count", regions=["l_cheek", "r_cheek"], log=True),
    "pore_grade": dict(column="pore_grade", regions=["l_cheek", "r_cheek"], log=False),
    "wrinkle_grade": dict(column="wrinkle_grade", regions=["forehead", "glabella", "l_eye", "r_eye"], log=False),
    "wrinkle_ra": dict(column="wrinkle_ra", regions=["l_eye", "r_eye"], log=False),
    "pigment_grade": dict(column="pigment_grade", regions=["forehead", "l_cheek", "r_cheek"], log=False),
    "acne_count": dict(column="acne_count", regions=["forehead", "l_cheek", "r_cheek"], log=True),
    "age": dict(column="age", regions=REGIONS, log=False),
}
TASK_NAMES = list(TASKS)

CFG = dict(
    views=["F"],                 # 실험용은 정면만. 최종 학습은 ["F", "L", "R"]
    min_skin_px=20_000,          # 피부가 이보다 적은 크롭은 제외 (가려졌거나 추출이 이상한 경우)
    mirror_right=True,           # 오른쪽 부위를 좌우 반전해 왼쪽과 방향을 맞춘다
    primary_task="pore_count",   # 설정을 비교할 때 기준으로 삼는 항목
    detached_tasks=(),           # 백본에 영향을 주지 않을 항목 (예: ("age",))
    # 모델
    backbone="convnext_tiny.fb_in22k_ft_in1k",
    freeze_stages=2, head_hidden=256, head_act="gelu", dropout=0.2,
    # 학습
    folds=5, epochs=15, patience=4, batch_size=8,
    lr_backbone=2e-5, lr_head=2e-4, weight_decay=0.05,
    # 증강 (흐림·강한 색 변화는 모공·색소 신호를 망가뜨려서 쓰지 않는다)
    aug_rotate=10, aug_scale=0.05, aug_bright=0.1,
    num_workers=4, seed=42,
)
PIXEL_MEAN, PIXEL_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet 사전학습 기준

print(f"장치 {DEVICE} (AMP {'사용' if USE_AMP else '미사용'}) | 백본 {CFG['backbone']}")
print(f"부위 {REGIONS}")
print(f"항목 {TASK_NAMES}")

## ② 데이터

저장된 크롭은 원본 해상도다. 읽은 뒤 **증강 → 배율 통일 → 캔버스** 순서로 처리한다.
회전을 원본 크기에서 하는 이유는, 이미 작아진 이미지에서 회전하면 질감이 절반으로 뭉개지기 때문이다.

In [ ]:
def load_rows(cfg):
    """index.csv와 정답을 합쳐 크롭 목록을 만든다. 테스트 165명은 따로 뺀다."""
    labels = {}
    with open(LABEL_PATH, newline="") as f:
        for row in csv.DictReader(f):
            labels[(row["subject"], row["region"])] = {
                task: float(row[spec["column"]])
                for task, spec in TASKS.items()
                if row[spec["column"]] and row["region"] in spec["regions"]
            }

    test_subjects = {s["subject_id"]
                     for s in json.loads((AILAB_DIR / "test_subjects_165.json").read_text())["subjects"]}

    rows, test_rows = [], []
    with open(INDEX_PATH, newline="") as f:
        for row in csv.DictReader(f):
            if row["status"] != "ok" or row["region"] not in REGIONS or row["view"] not in cfg["views"]:
                continue
            if int(row["skin_px"]) < cfg["min_skin_px"]:
                continue
            targets = labels.get((row["subject"], row["region"]))
            if not targets:
                continue
            sample = dict(subject=row["subject"], device=row["device"], view=row["view"], region=row["region"],
                          file=row["file"], face_h=float(row["face_h"]), targets=targets)
            (test_rows if row["subject"] in test_subjects else rows).append(sample)
    return rows, test_rows


def transform(task, value):
    """치우친 값은 log를 씌워 분포를 고르게 만든다."""
    return float(np.log1p(value)) if TASKS[task]["log"] else float(value)


def inverse(task, value):
    return float(np.expm1(value)) if TASKS[task]["log"] else float(value)


def target_stats(rows):
    """항목마다 평균·표준편차. 검증 정보가 새지 않게 학습 rows만 넣을 것.

    항목별로만 구한다. 한 항목 안에서는 부위가 달라도 단위가 같고(주름 등급은 어디든 0~6),
    부위별 차이(볼이 이마보다 색소가 짙다)는 모델이 배워야 할 실제 정보다.
    반면 항목 간 단위 차이는 549배까지 나서, 표준화하지 않으면 큰 항목이 손실을 독차지한다.
    """
    collected = defaultdict(list)
    for row in rows:
        for task, value in row["targets"].items():
            collected[task].append(transform(task, value))
    return {task: (float(np.mean(v)), float(np.std(v)) or 1.0) for task, v in collected.items()}


class RegionDataset(Dataset):
    """크롭 → 이미지 텐서, 표준화된 정답 (항목 수,), 마스크 (항목 수,), 행 번호"""

    def __init__(self, rows, stats, cfg, train=False):
        self.rows, self.stats, self.cfg, self.train = rows, stats, cfg, train
        self._rng = None

    def __len__(self):
        return len(self.rows)

    @property
    def rng(self):
        # 워커마다 다른 난수를 쓰도록 첫 사용 시점에 만든다 (전부 같은 seed면 증강이 겹친다)
        if self._rng is None:
            worker = get_worker_info()
            self._rng = np.random.default_rng(self.cfg["seed"] + (worker.id if worker else 0))
        return self._rng

    def __getitem__(self, i):
        row = self.rows[i]
        rgba = cv2.imread(str(CROP_DIR / row["file"]), cv2.IMREAD_UNCHANGED)
        if rgba is None:
            raise FileNotFoundError(f"크롭을 읽지 못했다: {CROP_DIR / row['file']}")
        crop = cv2.cvtColor(rgba[..., :3], cv2.COLOR_BGR2RGB)
        mask = (rgba[..., 3] > 0).astype(np.uint8)

        if self.cfg["mirror_right"] and row["region"].startswith("r_"):
            crop, mask = crop[:, ::-1].copy(), mask[:, ::-1].copy()
        if self.train:
            crop, mask = self.augment(crop, mask)
        crop, mask = normalize_crop(crop, mask, row["face_h"], row["region"])

        image = torch.from_numpy(crop).permute(2, 0, 1).float() / 255
        image = (image - torch.tensor(PIXEL_MEAN)[:, None, None]) / torch.tensor(PIXEL_STD)[:, None, None]
        # 정규화 후 0은 '평균 색'이다. 다각형 밖을 0으로 두면 새까만 테두리 대신 무난한 회색이 된다
        image *= torch.from_numpy(mask).float()

        targets, valid = torch.zeros(len(TASK_NAMES)), torch.zeros(len(TASK_NAMES))
        for t, task in enumerate(TASK_NAMES):
            if task in row["targets"]:
                mean, std = self.stats[task]
                targets[t] = (transform(task, row["targets"][task]) - mean) / std
                valid[t] = 1
        return image, targets, valid, i

    def augment(self, crop, mask):
        cfg = self.cfg
        h, w = crop.shape[:2]
        angle = self.rng.uniform(-cfg["aug_rotate"], cfg["aug_rotate"])
        scale = self.rng.uniform(1 - cfg["aug_scale"], 1 + cfg["aug_scale"])
        matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
        crop = cv2.warpAffine(crop, matrix, (w, h), flags=cv2.INTER_LINEAR)
        mask = cv2.warpAffine(mask, matrix, (w, h), flags=cv2.INTER_NEAREST)
        if cfg["aug_bright"] and self.rng.random() < 0.5:
            factor = self.rng.uniform(1 - cfg["aug_bright"], 1 + cfg["aug_bright"])
            crop = np.clip(crop * factor, 0, 255).astype(np.uint8)
        return crop, mask


class RegionBatchSampler(Sampler):
    """한 배치를 같은 부위끼리만 묶는다.

    부위마다 캔버스 크기가 달라서(이마 288x608, 볼 576x512) 섞으면 텐서 하나로 못 담는다.
    배치 안은 같은 부위, 배치 순서는 섞어서 학습이 한쪽 부위에 쏠리지 않게 한다.
    """

    def __init__(self, rows, batch_size, shuffle, drop_last, seed):
        self.by_region = defaultdict(list)
        for i, row in enumerate(rows):
            self.by_region[row["region"]].append(i)
        self.batch_size, self.shuffle, self.drop_last, self.seed = batch_size, shuffle, drop_last, seed
        self.epoch = 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        batches = []
        for indices in self.by_region.values():
            indices = list(indices)
            if self.shuffle:
                rng.shuffle(indices)
            for start in range(0, len(indices), self.batch_size):
                batch = indices[start:start + self.batch_size]
                if len(batch) == self.batch_size or not self.drop_last:
                    batches.append(batch)
        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return sum(len(v) // self.batch_size if self.drop_last else int(np.ceil(len(v) / self.batch_size))
                   for v in self.by_region.values())


def make_loader(rows, stats, cfg, train):
    dataset = RegionDataset(rows, stats, cfg, train=train)
    sampler = RegionBatchSampler(rows, cfg["batch_size"], shuffle=train, drop_last=train, seed=cfg["seed"])
    return DataLoader(dataset, batch_sampler=sampler, num_workers=cfg["num_workers"],
                      persistent_workers=cfg["num_workers"] > 0, pin_memory=DEVICE == "cuda")


def split_folds(rows, folds):
    """사람 단위로 나눈다. 같은 사람의 사진은 정답이 같아서 섞이면 점수가 부풀려진다."""
    groups = [r["subject"] for r in rows]
    return [([rows[i] for i in tr], [rows[i] for i in va])
            for tr, va in GroupKFold(n_splits=folds).split(rows, groups=groups)]

In [ ]:
rows, test_rows = load_rows(CFG)
folds = split_folds(rows, CFG["folds"])

print(f"학습 크롭 {len(rows):,}개 (사람 {len({r['subject'] for r in rows}):,}명)")
print(f"테스트 보관 {len(test_rows):,}개 (사람 {len({r['subject'] for r in test_rows}):,}명) — 어디에도 쓰지 않는다")
print("\n항목별 학습 칸 수:", {t: sum(t in r["targets"] for r in rows) for t in TASK_NAMES})
print("부위별 크롭 수:", {g: sum(r["region"] == g for r in rows) for g in REGIONS})
print("\n항목별 기준값 (평균, 표준편차):")
for task, (mean, std) in target_stats(rows).items():
    print(f"  {task:14s} {mean:8.3f} {std:8.3f}" + ("   ← log(1+x) 기준" if TASKS[task]["log"] else ""))

## ③ 학습·평가 함수

- **손실**: 항목 안에서 평균 → 항목끼리 평균. 칸 수가 많은 항목이 독차지하지 않게 한다.
- **평가**: 같은 (사람, 부위)의 여러 사진 예측을 평균한 뒤 순위 상관과 MAE를 잰다.

In [ ]:
def masked_loss(pred, target, valid, loss_fn):
    """정답이 있는 칸만 손실에 넣는다."""
    losses = [loss_fn(pred[valid[:, t] > 0, t], target[valid[:, t] > 0, t])
              for t in range(pred.shape[1]) if (valid[:, t] > 0).any()]
    return torch.stack(losses).mean() if losses else pred.sum() * 0


def predict(net, loader):
    net.eval()
    preds, valids, indices = [], [], []
    with torch.no_grad():
        for images, _, valid, index in loader:
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
                out = net(images.to(DEVICE))
            preds.append(out.float().cpu().numpy())
            valids.append(valid.numpy())
            indices.append(index.numpy())
    return np.concatenate(preds), np.concatenate(valids), np.concatenate(indices)


def evaluate(net, loader, rows, stats, return_predictions=False):
    """항목별 성능. 사람·부위 단위로 예측을 평균한 뒤 잰다."""
    preds, valids, indices = predict(net, loader)
    results, predictions = {}, {}
    for t, task in enumerate(TASK_NAMES):
        mean, std = stats[task]
        grouped, truth = defaultdict(list), {}
        for pred, valid, i in zip(preds[:, t], valids[:, t], indices):
            if not valid:
                continue
            row = rows[i]
            key = (row["subject"], row["region"])
            grouped[key].append(inverse(task, pred * std + mean))
            truth[key] = row["targets"][task]
        if len(grouped) < 10:
            continue
        keys = list(grouped)
        pred_mean = np.array([np.mean(grouped[k]) for k in keys])
        actual = np.array([truth[k] for k in keys])
        results[task] = dict(spearman=float(spearmanr(pred_mean, actual).statistic),
                             pearson=float(pearsonr(pred_mean, actual).statistic),
                             mae=float(np.abs(pred_mean - actual).mean()), n=len(keys))
        predictions[task] = (keys, pred_mean, actual)
    return (results, predictions) if return_predictions else results


def mean_spearman(results):
    return float(np.mean([r["spearman"] for r in results.values()])) if results else 0.0


def train_fold(train_rows, valid_rows, cfg, epochs=None, verbose=True, save_path=None):
    """한 fold 학습. 검증 점수가 가장 좋았던 시점의 결과를 돌려준다."""
    epochs = epochs or cfg["epochs"]
    stats = target_stats(train_rows)  # 검증 fold 정보는 쓰지 않는다
    train_loader = make_loader(train_rows, stats, cfg, train=True)
    valid_loader = make_loader(valid_rows, stats, cfg, train=False)

    torch.manual_seed(cfg["seed"])
    net = SkinModel(TASK_NAMES, backbone=cfg["backbone"], freeze_stages=cfg["freeze_stages"],
                    head_hidden=cfg["head_hidden"], head_act=cfg["head_act"], dropout=cfg["dropout"],
                    detached_tasks=cfg["detached_tasks"]).to(DEVICE)
    optimizer = torch.optim.AdamW(net.param_groups(cfg["lr_backbone"], cfg["lr_head"]),
                                  weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)
    loss_fn = nn.SmoothL1Loss()  # 이상치(모공 2000개 같은 경우)에 덜 휘둘린다

    best = dict(score=-1, epoch=0, results={}, history=[], stats=stats)
    bad_epochs = 0
    for epoch in range(1, epochs + 1):
        net.train()
        started, total = time.time(), 0.0
        for images, targets, valid, _ in train_loader:
            optimizer.zero_grad()
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
                loss = masked_loss(net(images.to(DEVICE)), targets.to(DEVICE), valid.to(DEVICE), loss_fn)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += loss.item()

        results = evaluate(net, valid_loader, valid_rows, stats)
        score = results.get(cfg["primary_task"], {}).get("spearman", 0)
        best["history"].append(dict(epoch=epoch, loss=total / len(train_loader), primary=score,
                                    mean=mean_spearman(results),
                                    **{f"{k}_mae": v["mae"] for k, v in results.items()}))
        if verbose:
            summary = "  ".join(f"{k[:9]} {v['spearman']:.3f}" for k, v in results.items())
            print(f"  epoch {epoch:2d} loss {total / len(train_loader):.4f} | {summary} | "
                  f"평균 {mean_spearman(results):.3f} ({time.time() - started:.0f}초)", flush=True)

        if score > best["score"]:
            best.update(score=score, epoch=epoch, results=results)
            if save_path:
                torch.save({"model": net.state_dict(), "stats": stats, "cfg": cfg}, save_path)
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                if verbose:
                    print(f"  {cfg['patience']}번 좋아지지 않아 중단")
                break
    return best


print(SkinModel(TASK_NAMES, pretrained=False).summary())

# A. 하이퍼파라미터 실험

**원칙**
- **한 번에 하나만 바꾼다.** 격자 탐색은 5항목 × 3가지 = 243번이라 불가능하다.
- **fold 0, 같은 seed로 고정**해 비교한다.
- 실험용은 정면만 + 1 fold + 적은 에폭 (설정당 약 1시간, RTX면 20분).

**영향이 큰 순서**: 학습률 > 동결 단계 > 증강 강도 > 헤드 구조.
헤드는 전체 파라미터의 0.7%라 대개 차이가 오차 범위 안에 들어간다. 그래서 맨 마지막이다.

In [ ]:
RESULT_PATH = OUT_DIR / "tuning.csv"
train_rows, valid_rows = folds[0]  # 실험은 항상 fold 0으로 고정
print(f"fold 0: 학습 {len(train_rows):,} / 검증 {len(valid_rows):,} "
      f"(사람 {len({r['subject'] for r in train_rows})} / {len({r['subject'] for r in valid_rows})})")


def run_experiment(name, epochs=6, **overrides):
    """CFG를 임시로 바꿔 fold 0을 학습하고 결과를 tuning.csv에 한 줄 추가한다."""
    cfg = {**CFG, **overrides}
    print(f"\n=== {name} === {overrides}")
    started = time.time()
    best = train_fold(train_rows, valid_rows, cfg, epochs=epochs)

    result = dict(name=name, when=datetime.now().strftime("%m-%d %H:%M"),
                  minutes=round((time.time() - started) / 60, 1), epochs=epochs, best_epoch=best["epoch"],
                  primary=round(best["score"], 4), mean=round(mean_spearman(best["results"]), 4),
                  **{task: round(r["spearman"], 4) for task, r in best["results"].items()},
                  **{k: str(v) for k, v in overrides.items()})

    old = list(csv.DictReader(open(RESULT_PATH, newline=""))) if RESULT_PATH.exists() else []
    fields = list(dict.fromkeys([f for r in old for f in r] + list(result)))
    with open(RESULT_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(old + [result])

    print(f"→ {name}: 주지표 {result['primary']:.3f}, 평균 {result['mean']:.3f} ({result['minutes']:.0f}분)")
    return result


def show_results():
    if not RESULT_PATH.exists():
        print("아직 실험 결과가 없다.")
        return []
    results = list(csv.DictReader(open(RESULT_PATH, newline="")))
    fixed = {"name", "when", "minutes", "epochs", "best_epoch", "primary", "mean"}
    print(f"{'실험':20s} {'주지표':>7s} {'평균':>7s} {'분':>5s}  변경한 설정")
    for r in results:
        changed = {k: v for k, v in r.items() if k not in fixed and k not in TASK_NAMES and v}
        print(f"{r['name']:20s} {float(r['primary']):7.3f} {float(r['mean']):7.3f} "
              f"{float(r['minutes']):5.0f}  {changed}")
    return results

### 0단계. 오차 범위 재기 (가장 먼저)

검증이 160명이라 우연히 0.02~0.03씩 흔들린다. 같은 설정을 seed만 바꿔 두 번 돌려
그 폭을 먼저 확인해야 "이 설정이 더 낫다"를 판단할 수 있다.

In [ ]:
run_experiment("기준", epochs=6)
run_experiment("기준(seed 43)", epochs=6, seed=43)

results = show_results()
if len(results) >= 2:
    gap = abs(float(results[-1]["primary"]) - float(results[-2]["primary"]))
    print(f"\nseed만 달라서 생긴 차이: {gap:.3f} → 이보다 작은 차이는 의미 없다고 본다.")

### 1단계. 학습률 (영향이 가장 큼)

In [ ]:
run_experiment("학습률 낮게", epochs=6, lr_backbone=1e-5, lr_head=1e-4)
run_experiment("학습률 높게", epochs=6, lr_backbone=5e-5, lr_head=5e-4)
show_results();

### 2단계. 백본을 얼마나 얼릴지

`0`=거의 전부 학습, `2`=앞 4.4% 고정(지금), `3`=앞 44% 고정.
데이터가 800명뿐이라 많이 얼리는 쪽이 유리할 수 있다.

In [ ]:
run_experiment("동결 없음", epochs=6, freeze_stages=0)
run_experiment("동결 3단계", epochs=6, freeze_stages=3)
show_results();

### 3단계. 증강 강도

약하면 외우고, 강하면 신호가 망가진다.

In [ ]:
run_experiment("증강 약하게", epochs=6, aug_rotate=5, aug_scale=0.02, aug_bright=0.05)
run_experiment("증강 세게", epochs=6, aug_rotate=20, aug_scale=0.10, aug_bright=0.20)
show_results();

### 4단계. 헤드 구조 (영향이 가장 작음)

In [ ]:
run_experiment("헤드 1층", epochs=6, head_hidden=0)
run_experiment("헤드 512", epochs=6, head_hidden=512)
run_experiment("드롭아웃 0.4", epochs=6, dropout=0.4)
show_results();

In [ ]:
results = show_results()
if results:
    names = [r["name"] for r in results]
    x = np.arange(len(names))
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
    axes[0].bar(x - 0.2, [float(r["primary"]) for r in results], 0.4, label=f"주지표({CFG['primary_task']})")
    axes[0].bar(x + 0.2, [float(r["mean"]) for r in results], 0.4, label="전체 평균")
    axes[0].axhline(float(results[0]["primary"]), color="red", ls="--", lw=1, label="기준 실험")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=30, ha="right")
    axes[0].set_ylabel("Spearman")
    axes[0].legend()
    axes[0].grid(alpha=.3, axis="y")

    tasks = [t for t in TASK_NAMES if t in results[0]]
    for r in results:
        axes[1].plot(tasks, [float(r[t]) for t in tasks], marker="o", label=r["name"])
    axes[1].set_title("항목별 성능")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].grid(alpha=.3)
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

# B. 최종 학습 (5-fold)

실험에서 정한 설정을 `FINAL`에 넣고 돌린다. 각도 3개를 모두 쓰므로 크롭이 2배가 된다.
맥 기준 13시간, RTX 3070 Ti 기준 3~4시간.

In [ ]:
FINAL = {**CFG, "views": ["F", "L", "R"], "epochs": 15}
FOLDS_TO_RUN = 5

final_rows, final_test_rows = load_rows(FINAL)
print(f"학습 크롭 {len(final_rows):,}개 / 테스트 {len(final_test_rows):,}개")

fold_results = []
for fold, (tr, va) in enumerate(split_folds(final_rows, FINAL["folds"])):
    if fold >= FOLDS_TO_RUN:
        break
    print(f"\n[fold {fold}] 학습 {len(tr):,} / 검증 {len(va):,}")
    fold_results.append(train_fold(tr, va, FINAL, save_path=OUT_DIR / f"fold{fold}.pt"))

print("\n=== fold 요약 (사람 단위 순위 상관) ===")
for task in TASK_NAMES:
    scores = [b["results"][task]["spearman"] for b in fold_results if task in b["results"]]
    if scores:
        print(f"  {task:14s} {np.mean(scores):.3f} ± {np.std(scores):.3f}   {[round(s, 3) for s in scores]}")

# C. 테스트 평가 + ailabtools 비교

학습·검증에 한 번도 쓰지 않은 165명이다. 같은 사람들을 상용 API로도 돌려놨으므로
**같은 정답 기준으로 순위 상관을 비교**한다. 단위가 달라서(우리는 기기 측정값, ailabtools는 자체 기준)
오차로는 비교할 수 없다.

In [ ]:
checkpoints = sorted(OUT_DIR.glob("fold*.pt"))
print(f"불러온 체크포인트: {[p.name for p in checkpoints]}")

all_predictions = []
for path in checkpoints:
    saved = torch.load(path, map_location=DEVICE, weights_only=False)
    cfg = saved["cfg"]
    net = SkinModel(TASK_NAMES, backbone=cfg["backbone"], pretrained=False, freeze_stages=0,
                    head_hidden=cfg["head_hidden"], head_act=cfg["head_act"], dropout=cfg["dropout"]).to(DEVICE)
    net.load_state_dict(saved["model"])
    loader = make_loader(final_test_rows, saved["stats"], cfg, train=False)
    _, predictions = evaluate(net, loader, final_test_rows, saved["stats"], return_predictions=True)
    all_predictions.append(predictions)
    print(f"  {path.name} 완료")

# 항목별로 fold 예측을 평균 (앙상블)
ensemble = {}
for task in TASK_NAMES:
    parts = [p[task] for p in all_predictions if task in p]
    if not parts:
        continue
    keys, actual = parts[0][0], parts[0][2]
    pred = np.mean([p[1] for p in parts], axis=0)
    ensemble[task] = dict(keys=keys, pred=pred, actual=actual,
                          spearman=float(spearmanr(pred, actual).statistic),
                          pearson=float(pearsonr(pred, actual).statistic),
                          mae=float(np.abs(pred - actual).mean()))

print(f"\n{'항목':16s} {'스피어맨':>9s} {'피어슨':>8s} {'MAE':>10s} {'개수':>6s}")
for task, r in ensemble.items():
    print(f"  {task:14s} {r['spearman']:9.3f} {r['pearson']:8.3f} {r['mae']:10.1f} {len(r['keys']):6d}")

In [ ]:
# ailabtools 결과 불러오기
ailab = {}
for sid in {s["subject_id"] for s in json.loads((AILAB_DIR / "test_subjects_165.json").read_text())["subjects"]}:
    path = AILAB_DIR / "raw" / f"{sid}.json"
    if path.exists():
        ailab[sid] = json.loads(path.read_text())["result"]

# (우리 항목, 부위) → ailabtools의 대응 값
AILAB_FIELDS = {
    ("pore_count", "l_cheek"): lambda r: r["enlarged_pore_count"]["left_cheek_count"],
    ("pore_count", "r_cheek"): lambda r: r["enlarged_pore_count"]["right_cheek_count"],
    ("pore_grade", "l_cheek"): lambda r: r["pores_left_cheek"]["value"],
    ("pore_grade", "r_cheek"): lambda r: r["pores_right_cheek"]["value"],
    ("wrinkle_grade", "forehead"): lambda r: r["forehead_wrinkle_severity"]["value"],
    ("wrinkle_grade", "glabella"): lambda r: r["glabella_wrinkle_severity"]["value"],
    ("wrinkle_grade", "l_eye"): lambda r: r["left_crows_feet_severity"]["value"],
    ("wrinkle_grade", "r_eye"): lambda r: r["right_crows_feet_severity"]["value"],
    ("wrinkle_ra", "l_eye"): lambda r: r["left_crows_feet_severity"]["value"],
    ("wrinkle_ra", "r_eye"): lambda r: r["right_crows_feet_severity"]["value"],
    **{("age", region): (lambda r: r["skin_age"]["value"]) for region in REGIONS},
}

print(f"{'항목':16s} {'우리 모델':>10s} {'ailabtools':>11s} {'비교 개수':>9s}")
comparison = {}
for task, r in ensemble.items():
    pairs = [(our, r["actual"][i], AILAB_FIELDS[(task, region)](ailab[subject]))
             for i, ((subject, region), our) in enumerate(zip(r["keys"], r["pred"]))
             if (task, region) in AILAB_FIELDS and subject in ailab]
    if len(pairs) < 20:
        print(f"  {task:14s} {r['spearman']:10.3f} {'대응 없음':>11s}")
        continue
    ours, actual, theirs = map(np.array, zip(*pairs))
    comparison[task] = (float(spearmanr(ours, actual).statistic), float(spearmanr(theirs, actual).statistic))
    print(f"  {task:14s} {comparison[task][0]:10.3f} {comparison[task][1]:11.3f} {len(pairs):9d}")

if comparison:
    tasks = list(comparison)
    x = np.arange(len(tasks))
    plt.figure(figsize=(9, 4))
    plt.bar(x - 0.2, [comparison[t][0] for t in tasks], 0.4, label="우리 모델")
    plt.bar(x + 0.2, [comparison[t][1] for t in tasks], 0.4, label="ailabtools")
    plt.xticks(x, tasks, rotation=20)
    plt.ylabel("Spearman (같은 정답 기준)")
    plt.title("테스트 165명 — 상용 API와 비교")
    plt.legend()
    plt.grid(alpha=.3, axis="y")
    plt.tight_layout()
    plt.show()

# D. 서버 배포용 내보내기

앱이 크롭을 만들어 보내고 서버는 추론만 하는 구조다. 서버에 올릴 것은 두 개다.

- **`skin.onnx`** — 가중치 + 연산 그래프. `model.py` 없이 ONNX Runtime만으로 실행된다
- **`meta.json`** — 예측을 원래 단위로 되돌릴 기준(stats)과 입력 규격. 가중치는 없다 (수 KB)

ONNX Runtime이 PyTorch CPU보다 5배 빠르다 (6부위 2.2초 → 0.4초).

**배포용 모델은 보통 전체 데이터로 다시 학습한다.** 교차검증은 설정과 에폭 수를 고르는 용도이고,
5개를 앙상블하면 메모리·시간이 5배인데 성능 향상은 1~2%다.
아래 `DEPLOY_CHECKPOINT`를 배포할 체크포인트로 지정하면 된다.

In [ ]:
DEPLOY_CHECKPOINT = OUT_DIR / "fold0.pt"   # 배포할 체크포인트
EXPORT_DIR = OUT_DIR / "deploy"
EXPORT_DIR.mkdir(exist_ok=True)

saved = torch.load(DEPLOY_CHECKPOINT, map_location="cpu", weights_only=False)
cfg = saved["cfg"]
net = SkinModel(TASK_NAMES, backbone=cfg["backbone"], pretrained=False, freeze_stages=0,
                head_hidden=cfg["head_hidden"], head_act=cfg["head_act"], dropout=cfg["dropout"])
net.load_state_dict(saved["model"])
net.eval()

# 부위마다 입력 크기가 달라서 세로·가로를 고정하지 않고 내보낸다.
# ConvNeXt는 합성곱 + 전역 평균 풀링이라 크기가 달라도 항상 768개 특징이 나온다.
onnx_path = EXPORT_DIR / "skin.onnx"
torch.onnx.export(
    net, torch.randn(1, 3, 576, 512), str(onnx_path),
    input_names=["crop"], output_names=["pred"],
    dynamic_axes={"crop": {0: "batch", 2: "height", 3: "width"}, "pred": {0: "batch"}},
    opset_version=17,
)

meta = {
    "version": datetime.now().strftime("%Y-%m-%d"),
    "tasks": TASK_NAMES,
    "regions": REGIONS,
    # 예측값 → 원래 단위: log면 expm1(pred * std + mean), 아니면 pred * std + mean
    "stats": {task: {"mean": saved["stats"][task][0], "std": saved["stats"][task][1],
                     "log": TASKS[task]["log"], "regions": TASKS[task]["regions"]}
              for task in TASK_NAMES if task in saved["stats"]},
    # 서버가 크롭을 모델 입력으로 만들 때 지켜야 하는 규격
    "input": {
        "canvas": {region: list(CANVAS[region]) for region in REGIONS},   # (가로, 세로)
        "target_face_height": face_regions.TARGET_FACE_HEIGHT,
        "mirror_right": cfg["mirror_right"],     # r_ 부위는 좌우 반전해서 넣는다
        "pixel_mean": list(PIXEL_MEAN),
        "pixel_std": list(PIXEL_STD),
        "mask_fill": 0,                          # 다각형 밖은 정규화 '후' 0 (= 평균 색)
    },
    "train": {"backbone": cfg["backbone"], "views": cfg["views"], "epochs": saved["cfg"]["epochs"],
              "checkpoint": DEPLOY_CHECKPOINT.name},
}
(EXPORT_DIR / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))

print(f"skin.onnx  {onnx_path.stat().st_size / 1e6:.0f} MB")
print(f"meta.json  {(EXPORT_DIR / 'meta.json').stat().st_size / 1024:.1f} KB")

In [ ]:
# 내보낸 ONNX가 PyTorch와 같은 값을 내는지, 부위별 크기가 다 되는지 확인
import onnxruntime as ort

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print(f"{'부위':10s} {'입력':>12s} {'torch와 차이':>12s} {'시간':>8s}")
total = 0
for region in REGIONS:
    width, height = CANVAS[region]
    x = np.random.randn(1, 3, height, width).astype(np.float32)
    with torch.no_grad():
        reference = net(torch.from_numpy(x)).numpy()
    started = time.perf_counter()
    output = session.run(None, {"crop": x})[0]
    elapsed = (time.perf_counter() - started) * 1000
    total += elapsed
    print(f"  {region:8s} {width}x{height:<8} {np.abs(output - reference).max():12.6f} {elapsed:7.0f} ms")
print(f"\n6부위 합계 {total / 1000:.1f}초 (사진 3장이면 14크롭이라 약 {total * 14 / 6 / 1000:.1f}초)")

print("""
서버에 올릴 것
  deploy/skin.onnx   가중치 + 연산 그래프 (model.py 불필요)
  deploy/meta.json   stats + 입력 규격

서버가 할 일
  1. 앱에서 받은 부위 크롭 + 얼굴 세로 길이
  2. meta.json 규격대로 정규화 (배율 통일 → 캔버스 → r_ 부위 반전 → 픽셀 정규화)
  3. session.run(None, {"crop": x}) → (1, 7)
  4. stats로 원래 단위 복원 → 같은 부위끼리 평균 → 최종 점수
""")